# Finite-ground CPW Cross Section — Q2D Extraction

## Setup and Imports

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    MatrixRunControl,
    PdkMaterial,
    Q2dConductorSpec,
    Q2dRectangleSpec,
    Q2dSpec,
    prepare_handoff,
    resolve_results,
)

import orpen_sc_pdk
from orpen_sc_pdk.materials import get_material_records

orpen_sc_pdk.activate()

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "prepare_handoff"  # prepare_handoff | run | analyze_handoff
RUN_ID = "cpw_finite_ground_q2d"
OUTPUT_ROOT = Path("notebooks/.artifacts/CrossSectionSimulation/CpwFiniteGround")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = RUN_DIR

## Create Simulation Component / Coupon

In [ ]:
signal_width_um = 10.0
gap_um = 10.0
ground_width_um = 40.0
metal_thickness_um = 0.2
substrate_thickness_um = 500.0
cross_section = gf.Component()
cross_section.add_polygon(
    [(-100.0, -500.0), (100.0, -500.0), (100.0, 0.0), (-100.0, 0.0)],
    layer=(201, 0),
)
cross_section.add_polygon([(-5.0, 0.0), (5.0, 0.0), (5.0, 0.2), (-5.0, 0.2)], layer=(1, 0))
cross_section.add_polygon([(-55.0, 0.0), (-15.0, 0.0), (-15.0, 0.2), (-55.0, 0.2)], layer=(2, 0))
cross_section.add_polygon([(15.0, 0.0), (55.0, 0.0), (55.0, 0.2), (15.0, 0.2)], layer=(2, 0))
cross_section.plot()

## Initialize AEDT Project / App

In [ ]:
aedt_version = "2024.2"
project_name = RUN_ID
design_name = "CpwFiniteGroundQ2d"

## Import GDS and Build the HFSS/Q3D/Q2D Model

In [ ]:
rectangles = (
    Q2dRectangleSpec("Substrate", (-100.0, -substrate_thickness_um), (200.0, 500.0), "Si"),
    Q2dRectangleSpec("Signal", (-signal_width_um / 2, 0.0), (signal_width_um, 0.2), "Nb"),
    Q2dRectangleSpec("GroundLeft", (-55.0, 0.0), (ground_width_um, 0.2), "Nb"),
    Q2dRectangleSpec("GroundRight", (15.0, 0.0), (ground_width_um, 0.2), "Nb"),
)

## Geometry Verification

In [ ]:
display(cross_section)

## Materials and Boundaries

In [ ]:
material_records = get_material_records()
materials = {
    material_id: PdkMaterial(
        material_id,
        material_records[material_id]["material_kind"],
        material_records[material_id]["is_superconducting"],
        material_records[material_id]["aedt_library_name"],
    )
    for material_id in ("vacuum", "Si", "Nb")
}
region_padding_um = (100.0, 100.0, 1000.0, 100.0)

## Ports / Nets / Excitations

In [ ]:
conductors = (
    Q2dConductorSpec("Signal", "SignalLine", ("Signal",), metal_thickness_um),
    Q2dConductorSpec(
        "Ground", "ReferenceGround", ("GroundLeft", "GroundRight"), metal_thickness_um
    ),
)

## Simulation Setup

In [ ]:
frequency_ghz = 6.0
maximum_passes = 3
spec = Q2dSpec(
    project_name=project_name,
    design_name=design_name,
    materials=materials,
    vacuum_material_id="vacuum",
    rectangles=rectangles,
    conductors=conductors,
    run_control=MatrixRunControl("Setup1", frequency_ghz, maximum_passes),
    region_padding_um=region_padding_um,
    aedt_version=aedt_version,
)

## Simulation Configuration

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = prepare_handoff(spec=spec, output_dir=RUN_DIR)
display(HANDOFF)

## Solve and Export

In [ ]:
if WORKFLOW_ACTION == "run":
    subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
RESULT = resolve_results(RETURNED_RUN_DIR) if WORKFLOW_ACTION == "analyze_handoff" else None
display(RESULT)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if RESULT is not None:
    display(RESULT.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if RESULT is not None:
    display(RESULT.simulation_benchmark())

## Save and Release AEDT

In [ ]:
display(RESULT.project_path if RESULT is not None else HANDOFF.archive_path)